# LLM Pipeline - Quick Start Notebook

This notebook demonstrates the core functionality of the LLM Pipeline project.

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '..')

## 1. Configuration

In [ ]:
from src.config import settings

print(f"Environment: {settings.environment}")
print(f"Base Model: {settings.model.base_model_name}")
print(f"Max Sequence Length: {settings.model.max_seq_length}")

## 2. Data Loading

In [ ]:
from src.data_loader import DataLoader, create_sample_dataset

# Create sample data for demonstration
sample_path = create_sample_dataset(num_samples=100)
print(f"Created sample data at: {sample_path}")

In [ ]:
# Load and preprocess data
loader = DataLoader()
dataset = loader.load_json(sample_path)
print(f"Loaded {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"\nSample: {dataset[0]}")

In [ ]:
# Preprocess and split
processed = loader.preprocess_dataset(dataset)
splits = loader.split_dataset(processed)

print(f"Train: {len(splits['train'])} samples")
print(f"Validation: {len(splits['validation'])} samples")
print(f"Test: {len(splits['test'])} samples")

## 3. Model Inference (Using Base Model)

In [ ]:
from src.inference import LLMInference

# Initialize inference engine
inference = LLMInference(
    task_type="classification",
    enable_monitoring=True,
)

# Get model info
info = inference.get_model_info()
print(f"Model Version: {info['model_version']}")
print(f"Device: {info['device']}")

In [ ]:
# Make a prediction
result = inference.predict(
    text="This is a great product! I love it.",
    return_all_scores=True,
)

print(f"Input: {result.input_text}")
print(f"Output: {result.output}")
print(f"Inference Time: {result.inference_time_ms:.2f} ms")

In [ ]:
# Batch prediction
texts = [
    "This is amazing!",
    "I'm disappointed with this purchase.",
    "It's okay, nothing special.",
]

results = inference.predict_batch(texts)

for r in results:
    print(f"Text: {r.input_text[:50]}...")
    print(f"  Prediction: {r.output}")
    print()

## 4. Monitoring

In [ ]:
from src.monitor import PredictionMonitor

monitor = PredictionMonitor()
metrics = monitor.get_metrics()

print("Monitoring Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

## 5. Fine-Tuning (Optional - Requires GPU)

In [ ]:
# Uncomment to run fine-tuning (requires GPU and more memory)
"""
from src.model import LLMFineTuner

# Tokenize data
tokenized = loader.tokenize_dataset(splits['train'])
tokenized_eval = loader.tokenize_dataset(splits['validation'])

# Initialize fine-tuner
tuner = LLMFineTuner(
    model_name="distilbert-base-uncased",
    task_type="classification",
    num_labels=3,  # positive, negative, neutral
)

# Load model
tuner.load_model()

# Train
results = tuner.train(
    train_dataset=tokenized,
    eval_dataset=tokenized_eval,
)

print(f"Training completed!")
print(f"Version: {results['version']}")
print(f"Metrics: {results['metrics']}")

# Save model
tuner.save_model()
"""

## 6. Cleanup

In [ ]:
# Unload model to free memory
inference.unload_model()
print("Model unloaded.")